<a href="https://colab.research.google.com/github/DianaBravoPerez/EDP-1/blob/main/matriz_fundamental2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Matriz fundamental**

In [1]:
import numpy as np
import random

Primero se construye la matriz de transición $P$ de tamaño $20 \times 20$.

Cada entrada $P_{ij}$ es la probabilidad de ir de la casilla $i$ a la casilla $j$ en un turno.

In [3]:
# destino de cada casilla (considerando serpientes y escaleras)
destino = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20]
destino[3]  = 11
destino[15] = 18
destino[17] = 9
destino[13] = 5

# matriz de transicion de 20x20
P = np.zeros((20, 20))

for i in range(1, 21):
    for dado in range(1, 7):
        j = i + dado
        if j > 20:
            j = i
        j = destino[j]
        P[i-1][j-1] += 1/6

print("Matriz P construida")

Matriz P construida


Ahora se extrae la submatriz $Q$, que solo incluye las transiciones entre los 19 estados transitorios (casillas 1 a 19).

In [4]:
Q = P[0:19, 0:19]

print("Submatriz Q (19x19):")
print(np.round(Q, 3))

Submatriz Q (19x19):
[[0.    0.167 0.    0.167 0.167 0.167 0.167 0.    0.    0.    0.167 0.
  0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.167 0.167 0.167 0.167 0.167 0.    0.    0.167 0.
  0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.167 0.167 0.167 0.167 0.167 0.167 0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.167 0.167 0.167 0.167 0.167 0.167 0.    0.
  0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.167 0.167 0.167 0.167 0.167 0.167 0.
  0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.167 0.167 0.167 0.167 0.167 0.167
  0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.167 0.    0.    0.167 0.167 0.167 0.167 0.167
  0.    0.    0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.167 0.    0.    0.    0.167 0.167 0.167 0.167
  0.    0.167 0.    0.    0.    0.    0.   ]
 [0.    0.    0.    0.    0.167 0.    0.  

Se calcula la matriz fundamental $N = (I - Q)^{-1}$

In [5]:
I = np.eye(19)
N = np.linalg.inv(I - Q)

print("Matriz fundamental N calculada")

Matriz fundamental N calculada


El número promedio de tiradas para terminar el juego desde la casilla $i$ es la suma de la fila $i$ de $N$:

$$t_i = \sum_j N_{ij}$$

In [6]:
t = N.sum(axis=1)

print("Número promedio de tiradas para terminar desde cada casilla:")
for i in range(19):
    print(f"  Casilla {i+1}: {t[i]:.2f} tiradas")

print()
print(f">>> Desde la casilla 1 (inicio): {t[0]:.2f} tiradas en promedio")

Número promedio de tiradas para terminar desde cada casilla:
  Casilla 1: 11.42 tiradas
  Casilla 2: 11.20 tiradas
  Casilla 3: 11.25 tiradas
  Casilla 4: 10.93 tiradas
  Casilla 5: 10.65 tiradas
  Casilla 6: 10.36 tiradas
  Casilla 7: 10.40 tiradas
  Casilla 8: 9.87 tiradas
  Casilla 9: 9.32 tiradas
  Casilla 10: 8.96 tiradas
  Casilla 11: 9.01 tiradas
  Casilla 12: 8.58 tiradas
  Casilla 13: 7.81 tiradas
  Casilla 14: 6.69 tiradas
  Casilla 15: 6.69 tiradas
  Casilla 16: 6.83 tiradas
  Casilla 17: 7.66 tiradas
  Casilla 18: 6.00 tiradas
  Casilla 19: 6.00 tiradas

>>> Desde la casilla 1 (inicio): 11.42 tiradas en promedio


### Verificación por simulación

In [7]:
def jugar():
    casilla = 1
    tiradas = 0
    while casilla < 20:
        dado = random.randint(1, 6)
        nueva = casilla + dado
        if nueva > 20:
            nueva = casilla
        casilla = destino[nueva]
        tiradas += 1
    return tiradas

resultados = [jugar() for _ in range(100000)]
print(f"Simulación:  {np.mean(resultados):.2f} tiradas en promedio")
print(f"Analítico:   {t[0]:.2f} tiradas en promedio")

Simulación:  11.44 tiradas en promedio
Analítico:   11.42 tiradas en promedio


---
## Problema 2: El ratón en el laberinto

El laberinto es una cuadrícula de 3×3. Las casillas están numeradas así:

```
[ 0 ][ 1 ][ 7 comida ]
[ 2 ][ 3 ][ 4        ]
[ 8 ][ 5 ][ 6        ]
  shock
```

El ratón empieza en la casilla 0 y se mueve aleatoriamente a sus lados con igual probabilidad

Los estados absorbentes son:
- Casilla 7: comida
- Casilla 8: shock

Se quiere saber cuál es la probabilidad de que el ratón llegue a la comida

Primero se define quiénes son los vecinos de cada casilla.

In [8]:
vecinos = {
    0: [1, 2],
    1: [0, 3, 7],
    2: [0, 3, 8],
    3: [1, 2, 4, 5],
    4: [3, 6, 7],
    5: [3, 6, 8],
    6: [4, 5],
    7: [7],
    8: [8],
}

print("Vecinos definidos")

Vecinos definidos


Se construye la matriz de transición $P$ de tamaño $9 \times 9$.

In [9]:
P2 = np.zeros((9, 9))

for i in range(9):
    for j in vecinos[i]:
        P2[i][j] = 1 / len(vecinos[i])

print("Matriz de transición P2:")
print(np.round(P2, 3))

Matriz de transición P2:
[[0.    0.5   0.5   0.    0.    0.    0.    0.    0.   ]
 [0.333 0.    0.    0.333 0.    0.    0.    0.333 0.   ]
 [0.333 0.    0.    0.333 0.    0.    0.    0.    0.333]
 [0.    0.25  0.25  0.    0.25  0.25  0.    0.    0.   ]
 [0.    0.    0.    0.333 0.    0.    0.333 0.333 0.   ]
 [0.    0.    0.    0.333 0.    0.    0.333 0.    0.333]
 [0.    0.    0.    0.    0.5   0.5   0.    0.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    1.    0.   ]
 [0.    0.    0.    0.    0.    0.    0.    0.    1.   ]]


Los estados transitorios son 0, 1, 2, 3, 4, 5, 6 y los absorbentes son 7 (comida) y 8 (shock)

Se extraen las submatrices $Q$ y $R$:
- $Q$: transiciones entre estados transitorios
- $R$: transiciones de transitorios a absorbentes

In [10]:
Q2 = P2[0:7, 0:7]
R2 = P2[0:7, 7:9]

print("Q2:")
print(np.round(Q2, 3))
print()
print("R2:")
print(np.round(R2, 3))

Q2:
[[0.    0.5   0.5   0.    0.    0.    0.   ]
 [0.333 0.    0.    0.333 0.    0.    0.   ]
 [0.333 0.    0.    0.333 0.    0.    0.   ]
 [0.    0.25  0.25  0.    0.25  0.25  0.   ]
 [0.    0.    0.    0.333 0.    0.    0.333]
 [0.    0.    0.    0.333 0.    0.    0.333]
 [0.    0.    0.    0.    0.5   0.5   0.   ]]

R2:
[[0.    0.   ]
 [0.333 0.   ]
 [0.    0.333]
 [0.    0.   ]
 [0.333 0.   ]
 [0.    0.333]
 [0.    0.   ]]


Se calcula la matriz fundamental $N = (I - Q)^{-1}$ y luego $B = N \cdot R$

Cada entrada $B_{ij}$ es la probabilidad de ser absorbido en el estado $j$, partiendo del estado $i$

In [13]:
I2 = np.eye(7)
N2 = np.linalg.inv(I2 - Q2)
B  = N2 @ R2

print("Probabilidades de absorción B = N·R:")
print()
print(f"{'Estado':>8} | {'P(comida)':>10} | {'P(shock)':>10}")
print("-" * 35)
for i in range(7):
    print(f"  {i:>6}   |  {B[i,0]:>8.4f}  |  {B[i,1]:>8.4f}")

print()
print(f">>> P(ratón llega a comida | inicio=0) = {B[0,0]:.4f}")

Probabilidades de absorción B = N·R:

  Estado |  P(comida) |   P(shock)
-----------------------------------
       0   |    0.5000  |    0.5000
       1   |    0.6667  |    0.3333
       2   |    0.3333  |    0.6667
       3   |    0.5000  |    0.5000
       4   |    0.6667  |    0.3333
       5   |    0.3333  |    0.6667
       6   |    0.5000  |    0.5000

>>> P(ratón llega a comida | inicio=0) = 0.5000


### Verificación por simulación

In [14]:
def simular_raton():
    casilla = 0
    while casilla not in [7, 8]:
        casilla = random.choice(vecinos[casilla])
    return casilla

llegadas = [simular_raton() for _ in range(100000)]
prob_sim = llegadas.count(7) / 100000

# Ensure B is defined within this cell's scope
# N2 and R2 are expected to be available from previous executions.
B  = N2 @ R2

print(f"Simulación:  P(comida) = {prob_sim:.4f}")
print(f"Analítico:   P(comida) = {B[0,0]:.4f}")

Simulación:  P(comida) = 0.5003
Analítico:   P(comida) = 0.5000
